# HR Data Quality & Exploratory Analysis


## 1. Load and inspect the dataset


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Update this path if your CSV is stored elsewhere
DATA_PATH = Path(r"C:\Users\sacha\Downloads\archive (2)\WA_Fn-UseC_-HR-Employee-Attrition.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. "
        "Place the CSV beside this notebook or update DATA_PATH."
    )

C:\Users\sacha\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
import pandas as pd 

df = pd.read_csv(DATA_PATH)
# Display the first five records
display(df.head())


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [3]:
# Check the size of the dataset
print(f"Number of records: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

# Inspect column names and data types
df.info()

Number of records: 1,470
Number of columns: 35
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 

## 2. Checking for missing values and duplicates


In [4]:
missing_summary = pd.DataFrame({
    "Missing values": df.isnull().sum()
})

# Show columns with missing values first
print(
    missing_summary
    .sort_values("Missing values", ascending=False)
)

print(f"Total missing values: {df.isnull().sum().sum()}")

                          Missing values
Age                                    0
StandardHours                          0
NumCompaniesWorked                     0
Over18                                 0
OverTime                               0
PercentSalaryHike                      0
PerformanceRating                      0
RelationshipSatisfaction               0
StockOptionLevel                       0
MonthlyIncome                          0
TotalWorkingYears                      0
TrainingTimesLastYear                  0
WorkLifeBalance                        0
YearsAtCompany                         0
YearsInCurrentRole                     0
YearsSinceLastPromotion                0
MonthlyRate                            0
MaritalStatus                          0
Attrition                              0
EmployeeCount                          0
BusinessTravel                         0
DailyRate                              0
Department                             0
DistanceFromHome

In [5]:
duplicate_rows = df.duplicated().sum()

# Check employee identifier uniqueness
duplicate_ids = df["EmployeeNumber"].duplicated().sum()
unique_ids = df["EmployeeNumber"].nunique()

print(f"Exact duplicate rows: {duplicate_rows}")
print(f"Duplicate employee IDs: {duplicate_ids}")
print(f"Unique employee IDs: {unique_ids:,}")

if duplicate_rows == 0 and duplicate_ids == 0:
    print("Result: No duplicate rows or employee IDs detected.")
else:
    print("Result: Duplicates require further investigation.")

Exact duplicate rows: 0
Duplicate employee IDs: 0
Unique employee IDs: 1,470
Result: No duplicate rows or employee IDs detected.


## 3. Data quality and business rule validation


In [6]:
# Define validation checks
quality_checks = {
    "Age below 18": (df["Age"] < 18).sum(),
    "Age above 100": (df["Age"] > 100).sum(),
    "Negative years at company": (df["YearsAtCompany"] < 0).sum(),
    "Negative total working years": (df["TotalWorkingYears"] < 0).sum(),
    "Negative years in current role": (df["YearsInCurrentRole"] < 0).sum(),
    "Negative years since last promotion": (df["YearsSinceLastPromotion"] < 0).sum(),
    "Negative years with current manager": (df["YearsWithCurrManager"] < 0).sum(),
    "Invalid attrition category": (~df["Attrition"].isin(["Yes", "No"])).sum()
}

quality_report = pd.DataFrame(
    list(quality_checks.items()),
    columns=["Validation check", "Issues found"]
)

quality_report["Result"] = quality_report["Issues found"].apply(
    lambda n: "PASS" if n == 0 else "REVIEW"
)

display(quality_report)

print(f"Total issues across these checks: {quality_report['Issues found'].sum()}")

,Validation check,Issues found,Result
0,Age below 18,0,PASS
1,Age above 100,0,PASS
2,Negative years at company,0,PASS
3,Negative total working years,0,PASS
4,Negative years in current role,0,PASS
5,Negative years since last promotion,0,PASS
6,Negative years with current manager,0,PASS
7,Invalid attrition category,0,PASS


Total issues across these checks: 0


In [7]:
## 5. Identify constant and non-informative columns


In [8]:
unique_counts = df.nunique(dropna=False).sort_values()

display(unique_counts.to_frame(name="Distinct values"))

constant_columns = [
    col for col in df.columns
    if df[col].nunique(dropna=False) == 1
]

print("Constant columns:", constant_columns)

,Distinct values
Over18,1
StandardHours,1
EmployeeCount,1
Gender,2
Attrition,2
PerformanceRating,2
OverTime,2
MaritalStatus,3
Department,3
BusinessTravel,3


Constant columns: ['EmployeeCount', 'Over18', 'StandardHours']


In [9]:

df_clean = df.drop(columns=constant_columns).copy()

total_employees = df_clean["EmployeeNumber"].nunique()

employees_marked_yes = df_clean.loc[
    df_clean["Attrition"] == "Yes", "EmployeeNumber"
].nunique()

attrition_percentage = (
    employees_marked_yes / total_employees * 100
    if total_employees > 0 else 0
)

average_age = df_clean["Age"].mean()
average_tenure = df_clean["YearsAtCompany"].mean()

print("WORKFORCE KPI SUMMARY")
print("-" * 32)
print(f"Total employees: {total_employees:,}")
print(f"Employees marked Attrition = Yes: {employees_marked_yes:,}")
print(f"Observed attrition proportion: {attrition_percentage:.2f}%")
print(f"Average age: {average_age:.1f} years")
print(f"Average tenure: {average_tenure:.1f} years")

WORKFORCE KPI SUMMARY
--------------------------------
Total employees: 1,470
Employees marked Attrition = Yes: 237
Observed attrition proportion: 16.12%
Average age: 36.9 years
Average tenure: 7.0 years


In [10]:
# Create realistic-looking synthetic employee IDs

df_clean.insert(
    0,
    "EmployeeID",
    [
        f"EMP-{i:05d}"
        for i in range(1, len(df_clean) + 1)
    ]
)

# Remove the original source identifier
df_clean = df_clean.drop(columns=["EmployeeNumber"])

# Display the first records
display(df_clean.head())

,EmployeeID,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,EMP-00001,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,...,3,1,0,8,0,1,6,4,0,5
1,EMP-00002,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,...,4,4,1,10,3,3,10,7,1,7
2,EMP-00003,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,...,3,2,0,7,3,3,0,0,0,0
3,EMP-00004,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,...,3,3,0,8,3,3,8,7,3,0
4,EMP-00005,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,...,3,4,1,6,3,3,2,2,2,2


In [11]:
df_clean

,EmployeeID,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,EMP-00001,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,...,3,1,0,8,0,1,6,4,0,5
1,EMP-00002,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,...,4,4,1,10,3,3,10,7,1,7
2,EMP-00003,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,...,3,2,0,7,3,3,0,0,0,0
3,EMP-00004,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,...,3,3,0,8,3,3,8,7,3,0
4,EMP-00005,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,...,3,4,1,6,3,3,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,EMP-01466,36,No,Travel_Frequently,884,Research & Development,23,2,Medical,3,...,3,3,1,17,3,3,5,2,0,3
1466,EMP-01467,39,No,Travel_Rarely,613,Research & Development,6,1,Medical,4,...,3,1,1,9,5,3,7,7,1,7
1467,EMP-01468,27,No,Travel_Rarely,155,Research & Development,4,3,Life Sciences,2,...,4,2,1,6,0,3,6,2,0,3
1468,EMP-01469,49,No,Travel_Frequently,1023,Sales,2,3,Medical,4,...,3,4,0,17,3,2,9,6,0,8


In [12]:
cleaned_path = r"C:\Users\sacha\Downloads\HR-Employee-data.csv"

df_clean.to_csv(
    cleaned_path,
    index=False,
    encoding="utf-8-sig"
)

print("Cleaned dataset saved successfully!")
print(cleaned_path)

Cleaned dataset saved successfully!
C:\Users\sacha\Downloads\HR-Employee-data.csv
